# M3 Hadoop MapReduce — VS Code / Jupyter NO HANG Version

This notebook uses local MapReduce only. It does **not** start Hadoop Streaming, so it will not stay pending forever.

Run all cells.

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
M3 Hadoop MapReduce runner — VS Code / Jupyter SAFE version.

This version does NOT try Hadoop Streaming first, because Hadoop can hang in
Jupyter/VS Code. It runs the exact same mapper2.py + reducer.py locally.

Required files:
    /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/mapper2.py
    /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/reducer.py

Input:
    /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset/input.txt

Output:
    /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs/m3_wordcount_output_local.txt
"""

from pathlib import Path
import subprocess
import sys
import shutil

PROJECT_ROOT = Path("/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow")
ASSIGNMENTS_DIR = PROJECT_ROOT / "Assignments"
CODEBOOK_DIR = ASSIGNMENTS_DIR / "Codebook"
DATASET_DIR = ASSIGNMENTS_DIR / "Dataset"
OUTPUTS_DIR = ASSIGNMENTS_DIR / "Outputs"

MAPPER = CODEBOOK_DIR / "mapper2.py"
REDUCER = CODEBOOK_DIR / "reducer.py"
INPUT_FILE = DATASET_DIR / "input.txt"
OUTPUT_FILE = OUTPUTS_DIR / "m3_wordcount_output_local.txt"


def ensure_folders() -> None:
    CODEBOOK_DIR.mkdir(parents=True, exist_ok=True)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Codebook folder: {CODEBOOK_DIR}")
    print(f"[INFO] Dataset folder:  {DATASET_DIR}")
    print(f"[INFO] Outputs folder:  {OUTPUTS_DIR}")


def ensure_sample_input() -> None:
    if not INPUT_FILE.exists():
        INPUT_FILE.write_text(
            "Hadoop is useful for big data.\n"
            "Hadoop MapReduce uses mapper and reducer files.\n"
            "This assignment runs mapper2.py and reducer.py.\n",
            encoding="utf-8",
        )
        print(f"[INFO] Created sample input file: {INPUT_FILE}")
    else:
        print(f"[INFO] Using existing input file: {INPUT_FILE}")


def ensure_mapper_reducer_exist() -> None:
    missing = []
    if not MAPPER.exists():
        missing.append(str(MAPPER))
    if not REDUCER.exists():
        missing.append(str(REDUCER))
    if missing:
        raise FileNotFoundError(
            "Missing required file(s):\n"
            + "\n".join(missing)
            + "\n\nPlease place mapper2.py and reducer.py in:\n"
            + str(CODEBOOK_DIR)
        )
    print(f"[INFO] Mapper found:  {MAPPER}")
    print(f"[INFO] Reducer found: {REDUCER}")


def clean_output() -> None:
    if OUTPUT_FILE.exists():
        OUTPUT_FILE.unlink()
        print("[INFO] Previous output removed.")


def run_command(command, input_text: str, timeout_seconds: int = 30):
    return subprocess.run(
        command,
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
        timeout=timeout_seconds,
    )


def run_local_mapreduce() -> None:
    print("[INFO] Running local MapReduce. This avoids Hadoop/Jupyter hanging.")
    input_text = INPUT_FILE.read_text(encoding="utf-8", errors="ignore")

    mapper_result = run_command([sys.executable, str(MAPPER)], input_text=input_text)
    if mapper_result.returncode != 0:
        raise RuntimeError("Mapper failed:\n" + mapper_result.stderr)

    # Hadoop sorts mapper output before reducer. We do the same locally.
    sorted_mapper_output = "\n".join(sorted(mapper_result.stdout.splitlines())) + "\n"

    reducer_result = run_command([sys.executable, str(REDUCER)], input_text=sorted_mapper_output)
    if reducer_result.returncode != 0:
        raise RuntimeError("Reducer failed:\n" + reducer_result.stderr)

    OUTPUT_FILE.write_text(reducer_result.stdout, encoding="utf-8")
    print(f"[SUCCESS] Output saved to:\n{OUTPUT_FILE}")


def display_output() -> None:
    print("\n================ WORD COUNT OUTPUT ================\n")
    if OUTPUT_FILE.exists():
        print(OUTPUT_FILE.read_text(encoding="utf-8", errors="ignore"))
    else:
        print("No output file found.")


def main() -> None:
    ensure_folders()
    ensure_sample_input()
    ensure_mapper_reducer_exist()
    clean_output()
    run_local_mapreduce()
    display_output()


if __name__ == "__main__":
    main()
else:
    main()


[INFO] Codebook folder: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook
[INFO] Dataset folder:  /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset
[INFO] Outputs folder:  /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs
[INFO] Using existing input file: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset/input.txt
[INFO] Mapper found:  /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/mapper2.py
[INFO] Reducer found: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/reducer.py
[INFO] Running local MapReduce. This avoids Hadoop/Jupyter hanging.
[SUCCESS] Output saved to:
/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs/m3_wordcount_output_local.txt

================ WORD COUNT OUTPUT ================

assignment	1
big	1
data	1
files	1
hadoop	2
mapper	1
mapper2	1
mapreduce	1
py	2
reducer	2
runs	1
useful	1
uses	1

